# Peer effects on a large network

For profiles $i=1,\ldots,N$ and players $t=1,\ldots,T$, $T=500$, let $d_t\in\{0,1\}$ and

$$
\pi_{it}(d_t,d_{-t}) = d_t\left(x_t'\beta + \alpha z_i + \nu_{it} + \delta \sum_{t'} w_{tt'}d_{t'}\right), \qquad \delta \ge 0.
$$

By Monderer and Shapley (1996), an exact potential is

$$
V_i(d) = \sum_t d_t(x_t'\beta + \alpha z_i + \nu_{it}) + \delta \sum_{t<t'} w_{tt'}d_td_{t'}.
$$

Estimate $(\beta,\delta)$ from one iid-shock profile. Then estimate $(\alpha,\beta,\delta)$ over a grid for the correlation parameter $\sigma$ with $N=20$ profiles.


In [1]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import breadth_first_order, maximum_flow

import combrum as cb

T = 500
AVG_DEGREE = 10.0
LINK_PROB = 0.75
GRAPH_SEED = 20260423

BETA0 = np.array([-1.0, -0.5, -1.0, 0.5])
DELTA0 = 0.20
ALPHA0 = 0.50
SIGMA0 = 0.50

IID_SEED = 49892
CORRELATED_SEED = 2718
IID_S = 50
CORRELATED_N = 20
CORRELATED_S = 20
TOL = 1e-3
MAX_ITERS = 1000


## Potential solver

For $\delta\ge0$, maximizing $V_i(d)$ is a minimum $s$-$t$ cut.


In [2]:
def min_cut_choice(linear, edge_i, edge_j, pair):
    n = linear.size

    row_sum = np.zeros(n)
    np.add.at(row_sum, edge_i, pair)
    net = -linear - row_sum

    magnitude = max(np.abs(net).max(initial=0.0), pair.max(initial=0.0))
    scale = 1.0 if magnitude == 0.0 else 10.0 ** (8 - np.floor(np.log10(magnitude)))
    net_i = np.round(net * scale).astype(np.int64)
    pair_i = np.round(pair * scale).astype(np.int64)

    source, sink = 0, 1
    nodes = np.arange(n) + 2
    choose = net_i < 0
    linked = pair_i > 0

    rows = np.r_[np.full(choose.sum(), source), nodes[~choose], edge_i[linked] + 2]
    cols = np.r_[nodes[choose], np.full((~choose).sum(), sink), edge_j[linked] + 2]
    vals = np.r_[-net_i[choose], net_i[~choose], pair_i[linked]]

    cap = csr_matrix((vals, (rows, cols)), shape=(n + 2, n + 2))
    residual = (cap - maximum_flow(cap, source, sink).flow).tocsr()
    residual.data[residual.data <= 0] = 0
    residual.eliminate_zeros()

    _, pred = breadth_first_order(residual, i_start=source, directed=True, return_predecessors=True)
    selected = pred != -9999
    selected[source] = True
    return selected[2:]


## Graph and covariates


In [3]:
rng = np.random.default_rng(GRAPH_SEED)
side = np.sqrt(T)
radius = np.sqrt(AVG_DEGREE / (LINK_PROB * np.pi))
positions = rng.uniform(0.0, side, size=(T, 2))
distance = np.sqrt(((positions[:, None, :] - positions[None, :, :]) ** 2).sum(axis=-1))

link_shocks = np.triu(rng.logistic(size=(T, T)), k=1)
link_shocks = link_shocks + link_shocks.T
W = (distance <= radius) & (np.log(LINK_PROB / (1.0 - LINK_PROB)) >= link_shocks)
np.fill_diagonal(W, 0)

EDGE_I, EDGE_J = np.nonzero(np.triu(W, k=1))
PAIR0 = np.full(EDGE_I.size, DELTA0)

degree = W.sum(axis=1, dtype=np.float64)
inv_sqrt_degree = np.zeros_like(degree)
np.divide(1.0, np.sqrt(degree), out=inv_sqrt_degree, where=degree > 0)
W_TILDE = inv_sqrt_degree[:, None] * W * inv_sqrt_degree[None, :]

X = np.column_stack(
    [
        rng.integers(0, 2, size=T),
        rng.integers(0, 2, size=T),
        rng.uniform(0.0, 1.0, size=T),
        rng.uniform(0.0, 1.0, size=T),
    ]
)

{"players": T, "edges": EDGE_I.size, "mean_degree": round(degree.mean(), 3)}


{'players': 500, 'edges': 2370, 'mean_degree': 9.48}

## Oracle and feature map


In [4]:
class PeerGame(cb.Oracle, cb.FeatureMap):
    def __init__(self, z, eta_est, sigma=0.0, *, etaW_est=None, with_alpha):
        self.z = z
        self.eps = eta_est if etaW_est is None else eta_est + sigma * etaW_est
        self.N, self.S, _ = self.eps.shape
        self.with_alpha = with_alpha
        self.K = BETA0.size + 1 + with_alpha

    def features_batch(self, ids, bundles, *, weights=None, aggregate=False):
        obs = ids % self.N
        sim = ids // self.N

        Phi = np.empty((ids.size, self.K))
        col = 0
        if self.with_alpha:
            Phi[:, 0] = self.z[obs] * bundles.sum(axis=1)
            col = 1
        Phi[:, col : col + BETA0.size] = np.einsum("nt,tk->nk", bundles, X, optimize=True)
        Phi[:, -1] = (bundles[:, EDGE_I] * bundles[:, EDGE_J]).sum(axis=1)
        eps = np.einsum("nt,nt->n", self.eps[obs, sim], bundles, optimize=True)

        if aggregate:
            return np.einsum("n,nk->k", weights, Phi, optimize=True), np.einsum("n,n->", weights, eps, optimize=True)
        return Phi, eps

    def price_batch(self, theta, local_ids):
        obs = local_ids % self.N
        sim = local_ids // self.N

        if self.with_alpha:
            alpha = theta[0]
            beta = theta[1 : 1 + BETA0.size]
        else:
            alpha = 0.0
            beta = theta[: BETA0.size]
        delta = theta[-1]
        pair = np.full(EDGE_I.size, delta)

        base = np.einsum("tk,k->t", X, beta, optimize=True)

        bundles = np.empty((local_ids.size, T), dtype=bool)
        payoffs = np.empty(local_ids.size)
        for r, (i, s) in enumerate(zip(obs, sim)):
            linear = base + alpha * self.z[i] + self.eps[i, s]
            bundle = min_cut_choice(linear, EDGE_I, EDGE_J, pair)
            bundles[r] = bundle
            payoffs[r] = (
                np.sum(linear, where=bundle)
                + delta * np.count_nonzero(bundle[EDGE_I] & bundle[EDGE_J])
            )
        return cb.DemandBatch.exact(local_ids, bundles, payoffs)


## Iid shocks, $N=1$


In [5]:
rng = np.random.default_rng(IID_SEED)
iid_z = np.zeros(1)
iid_eta_dgp = rng.standard_normal((1, T))
iid_eta_est = rng.standard_normal((1, IID_S, T))
iid_y = min_cut_choice(
    np.einsum("tk,k->t", X, BETA0, optimize=True) + iid_eta_dgp[0],
    EDGE_I,
    EDGE_J,
    PAIR0,
)[None, :]

iid_theta0 = np.r_[BETA0, [DELTA0]]
iid_params = cb.Parameters({"beta": (-5.0, 5.0, BETA0.size), "delta": (0.0, 5.0, 1)})
iid_game = PeerGame(iid_z, iid_eta_est, with_alpha=False)
iid_model = cb.Model(iid_game, iid_params)
iid_data = cb.Data(observed_bundles=iid_y, shocks=iid_game.eps, observables=np.arange(iid_game.N))

iid_fit = cb.estimate(
    iid_model,
    iid_data,
    master_backend="highs",
    tolerance=TOL,
    max_iterations=MAX_ITERS,
    activity=cb.ActivityConfig(
        label="peer-iid",
        level="summary",
        stdout=True,
    ),
)

{
    "N": iid_game.N,
    "S": iid_game.S,
    "active_players": iid_y.sum(),
    "iterations": iid_fit.metadata["iterations"],
    "theta_true": np.round(iid_theta0, 4).tolist(),
    "theta_hat": np.round(iid_fit.theta_hat, 4).tolist(),
    "theta_error": np.round(iid_fit.theta_hat - iid_theta0, 4).tolist(),
    "max_abs_error": round(np.max(np.abs(iid_fit.theta_hat - iid_theta0)), 4),
}


[peer-iid] row generation: N=1 S=50 K=5 agents=50 ranks=1 tol=1.000e-03 max_iter=1000


[peer-iid] done converged=yes reason=converged iters=31 gap=2.769e-04 cuts=1339 obj=7.204e+03 wall=0.86s


{'N': 1,
 'S': 50,
 'active_players': 238,
 'iterations': 31,
 'theta_true': [-1.0, -0.5, -1.0, 0.5, 0.2],
 'theta_hat': [-1.02, -0.5326, -1.0918, 0.602, 0.1977],
 'theta_error': [-0.02, -0.0326, -0.0918, 0.102, -0.0023],
 'max_abs_error': 0.102}

## Correlated shocks, $N=20$

$$
\nu_i(\sigma) = (I + \sigma \widetilde W)\eta_i, \qquad \sigma \in [0,1].
$$

We estimate the model over a grid of values for $\sigma$. The first point is a cold fit, while the remaining points reuse the same NSlack master. Only the cut RHS changes with $\sigma$, so combRUM rewrites those values and warm-solves the current relaxation.

In [6]:
rng = np.random.default_rng(CORRELATED_SEED)
corr_z = np.zeros(CORRELATED_N)
corr_z[CORRELATED_N // 2 :] = 1.0
corr_eta_dgp = rng.standard_normal((CORRELATED_N, T))
corr_etaW_dgp = np.einsum("ns,ts->nt", corr_eta_dgp, W_TILDE, optimize=True)
corr_eta_est = rng.standard_normal((CORRELATED_N, CORRELATED_S, T))
corr_etaW_est = np.einsum("nst,ut->nsu", corr_eta_est, W_TILDE, optimize=True)

corr_y = np.empty((CORRELATED_N, T), dtype=bool)
for i, shock in enumerate(corr_eta_dgp + SIGMA0 * corr_etaW_dgp):
    corr_y[i] = min_cut_choice(np.einsum("tk,k->t", X, BETA0, optimize=True) + ALPHA0 * corr_z[i] + shock, EDGE_I, EDGE_J, PAIR0)

corr_theta0 = np.r_[[ALPHA0], BETA0, [DELTA0]]
corr_params = cb.Parameters(
    {"alpha": (-5.0, 5.0, 1), "beta": (-5.0, 5.0, BETA0.size), "delta": (0.0, 5.0, 1)}
)
observed_size2 = np.mean(corr_y.sum(axis=1) ** 2)

{"N": CORRELATED_N, "S": CORRELATED_S, "active_mean": round(corr_y.sum(axis=1).mean(), 3)}


{'N': 20, 'S': 20, 'active_mean': 315.6}

In [7]:
sigma_grid = np.linspace(0.2, .8, 50)
def rhs_sigma(row, sigma):
    i = row.agent_id % CORRELATED_N
    s = row.agent_id // CORRELATED_N
    return (
        np.sum(
            corr_eta_est[i, s] + sigma * corr_etaW_est[i, s],
            where=row.bundle,
        )
    )

driver = cb.PersistentMasterFit(
    corr_params,
    observables=np.arange(CORRELATED_N),
    observed_bundles=corr_y,
    transport=cb.SerialTransport(),
    config=cb.LoopConfig(max_iterations=MAX_ITERS),
    rhs_transform=rhs_sigma,
    master_backend="highs",
    tolerance=TOL,
)


sigma_rows = []
try:
    for k, sigma in enumerate(sigma_grid):
        game = PeerGame(
            corr_z,
            corr_eta_est,
            sigma,
            etaW_est=corr_etaW_est,
            with_alpha=True,
        )

        fit = driver.fit if k == 0 else driver.reevaluate
        result = fit(
            sigma,
            oracle=game,
            shocks=game.eps,
        )

        assert result.converged
        dual_bundles = result.dual.bundle_table[result.dual.bundle_row_ids]
        dual_pis = result.dual.pis.astype(np.longdouble)
        dual_sizes = dual_bundles.sum(axis=1, dtype=np.longdouble)
        dual_size2 = (
            dual_pis @ (dual_sizes * dual_sizes) / (game.N * game.S)
        )

        sigma_rows.append(
            {
                "sigma": sigma,
                "start": "cold" if k == 0 else "warm",
                "criterion": ((dual_size2 - observed_size2) / observed_size2)
                ** 2,
                "theta_hat": result.theta_hat,
                "dual_size2": dual_size2,
                "iterations": result.iterations,
                "active_cuts": result.n_active_cuts,
            }
        )
finally:
    driver.close()

In [8]:
param_names = ["alpha"] + [f"beta[{j}]" for j in range(BETA0.size)] + ["delta"]
sigma_best_id = np.argmin([row["criterion"] for row in sigma_rows])
sigma_best = sigma_rows[sigma_best_id]
theta_hat = sigma_best["theta_hat"]
theta_error = theta_hat - corr_theta0

print(f"Selected sigma: {sigma_best['sigma']:.2f}  (true {SIGMA0:.2f})")
print(f"Max parameter error: {np.max(np.abs(theta_error)):.4f}")
print()
print("Sigma grid")
print("  sigma  start  criterion  dual mean(size^2)  iterations  cuts")
for id, row in enumerate(sigma_rows):
    if np.abs(id - sigma_best_id) <= 5 or id == 0:
        mark = "*" if row is sigma_best else " "
        print(
            f"{mark} {row['sigma']:5.2f}  {row['start']:<5}"
            f"  {row['criterion']:9.2e}"
            f"  {row['dual_size2']:18.3f}"
            f"  {row['iterations']:10d}"
            f"  {row['active_cuts']:5d}"
        )
print(f"observed mean(size^2): {observed_size2:.3f}")

Selected sigma: 0.52  (true 0.50)
Max parameter error: 0.0314

Sigma grid
  sigma  start  criterion  dual mean(size^2)  iterations  cuts
   0.20  cold    1.82e-05          104646.208          28   9800
   0.46  warm    8.43e-07          104297.584           5  12526
   0.47  warm    5.33e-07          104277.996           4  12644
   0.48  warm    2.65e-07          104255.492           3  12770
   0.49  warm    1.20e-07          104237.967           5  12935
   0.51  warm    5.28e-08          104225.842           5  13031
*  0.52  warm    2.34e-09          104206.943           4  13162
   0.53  warm    1.01e-08          104191.418           4  13302
   0.54  warm    1.09e-07          104167.511           5  13419
   0.56  warm    3.76e-07          104137.974           5  13573
   0.57  warm    4.61e-07          104131.169           4  13675
   0.58  warm    7.36e-07          104112.501           4  13796
observed mean(size^2): 104201.900


In [9]:
print("Parameter recovery")
print("  parameter       true   estimate      error")
for name, true, estimate, error in zip(param_names, corr_theta0, theta_hat, theta_error):
    print(f"  {name:<9} {true:8.4f} {estimate:10.4f} {error:10.4f}")

Parameter recovery
  parameter       true   estimate      error
  alpha       0.5000     0.4946    -0.0054
  beta[0]    -1.0000    -0.9707     0.0293
  beta[1]    -0.5000    -0.4954     0.0046
  beta[2]    -1.0000    -0.9954     0.0046
  beta[3]     0.5000     0.5314     0.0314
  delta       0.2000     0.1951    -0.0049
